In [ ]:


import torch
import torch.nn as nn
import torchvision
import numpy as np
from matplotlib import pyplot as plt

from confidence.unsupervised.classic.SHE import SHETorchConfidence
from confidence.unsupervised.classic.dice import DICEConfidence
from confidence.unsupervised.classic.lof import LOFTorchConfidence
from model.classifier import Classifier, MyProgressBar
from utils.transforms.apply import grid_resample

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#look for experiment files in parents
import os
path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)


In [ ]:
experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")

In [ ]:
#get name of notebook this kernel runs
notebook_name = os.path.basename(globals()['__session__'])  # Get the name without extension
#remove the extension
notebook_name = os.path.splitext(notebook_name)[0]

In [ ]:
from its.transform import multi_transform
from utils.affine_transforms import AffineTransformations3D
from dataset.mnist_no_pil import NoPILMNIST, NoPILFashionMNIST, AffineTransformDataset

dataset = 'modelnet' # or 'fmnist'
batch_size = 32
#divide by 255 transform

from torch_geometric.datasets import ModelNet
from torch_geometric.transforms import NormalizeScale
from torch_geometric.transforms import SamplePoints

pre_transform, transform = NormalizeScale(), SamplePoints(1024)
data_path = os.path.join(experiment_files_path_data, "modelnet")

train_dataset_pre_val = ModelNet(data_path, '10', True, transform, pre_transform,force_reload=False)
#train val split
from torch.utils.data import random_split
train_dataset_pre, val_dataset_pre = random_split(train_dataset_pre_val, [0.9, 0.1])



test_dataset_pre_pre = ModelNet(data_path, '10', False, transform, pre_transform,force_reload=False)

from dataset.geometric_wrapper import GeometricLabelDatasetWrapper,GeometricsDatasetWrapper

dataset_train = GeometricsDatasetWrapper(train_dataset_pre)
dataset_val = GeometricsDatasetWrapper(val_dataset_pre)
dataset_test_pre = GeometricsDatasetWrapper(test_dataset_pre_pre)




transformations = [AffineTransformations3D.ROTATION.value]
domains = [(-torch.pi,torch.pi),]
n_samples = 17


#test simulated annealing
from utils.transformation_problem import TransformationProblem
from utils.transform_sequence import TransformSequence, create_parameter_sampler, create_sampler
from utils.transforms.apply import transform_3d_point_cloud

transform_seq= TransformSequence(transformations, domains, neighbour_hood_size =0.25,application_method=transform_3d_point_cloud,device="cpu",reflect=True,use_individual_param_correction=False)



sampler = create_sampler(transform_seq)

dataset_test = AffineTransformDataset(dataset_test_pre,sampler,return_transformation=False,batch_size=batch_size,resample_func=transform_3d_point_cloud)

transform_seq.to(device)


In [ ]:
dataset_train_transformed = AffineTransformDataset(dataset_train,sampler,return_transformation=False,batch_size=batch_size,resample_func=transform_3d_point_cloud)
train_loader_transformed = torch.utils.data.DataLoader(dataset_train_transformed, batch_size=batch_size, shuffle=True, num_workers=4,persistent_workers=True)

In [ ]:
train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=batch_size, shuffle=True, num_workers=4,persistent_workers=True)
val_loader = torch.utils.data.DataLoader(dataset_val, batch_size=batch_size, shuffle=False, num_workers=4,persistent_workers=True)
test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=batch_size, shuffle=True, num_workers=4,persistent_workers=True)

In [ ]:
dataset_val_transformed = AffineTransformDataset(dataset_val,sampler,return_transformation=False,batch_size=batch_size,resample_func=transform_3d_point_cloud)

val_loader_transformed = torch.utils.data.DataLoader(dataset_val_transformed, batch_size=batch_size, shuffle=True, num_workers=4,persistent_workers=True)

In [ ]:
modelpath = os.path.join(os.getcwd(), "model", f"{dataset}_metric.pth")

In [ ]:
import plotly.graph_objects as go
def plot_interactive_3d(points):
    fig = go.Figure(data=[go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        mode='markers',
        marker=dict(size=2)
    )])
    fig.update_layout(scene=dict(aspectmode='data'))
    fig.show()



In [ ]:
# Plot the 9th point cloud from the training dataset
plot_interactive_3d(dataset_train_transformed[140][0].numpy())


In [ ]:
from torch_scatter import scatter


class BatchPointNormalizer(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, data):
        pos = data.pos
        batch_idx = data.batch

        # Center points
        mean = scatter(pos, batch_idx, dim=0, reduce="mean")
        pos = pos - mean[batch_idx]

        # Scale points
        dist = (pos ** 2).sum(dim=-1).sqrt()
        max_dist = scatter(dist, batch_idx, dim=0, reduce="max")
        pos = pos / (max_dist[batch_idx].unsqueeze(-1) + 1e-8)

        data.pos = pos
        return data


from dataset.geometric_wrapper import BatchNormalizeScale
from model.pointnet_plus import PointNetPlus

model_not_normalized = PointNetPlus()
model = torch.nn.Sequential(BatchNormalizeScale(), model_not_normalized)
from dataset.geometric_wrapper import TensorGeometricModelWrapper
model = TensorGeometricModelWrapper(model)

In [ ]:
import pytorch_lightning as pl
import os
import torch
from torch import nn
from pytorch_lightning.callbacks import ModelCheckpoint

model_path = modelpath




# Check if model is already trained
if os.path.exists(model_path):
    print(f"Loading model from {model_path}")
    model.load_state_dict(torch.load(model_path))
else:
    print(f"Training model and will save to {model_path}")
    lightning_model = Classifier(model, optimizer_class =  torch.optim.AdamW, optimizer_params = {"lr": 1e-3})
    progress_bar = MyProgressBar()

    checkpoint_callback = ModelCheckpoint(
        monitor='val_acc',
        mode='max',
        save_top_k=1,
        verbose=True,
        dirpath=os.path.join(os.getcwd(), "model"),
        filename=f"{dataset}_val_acc_best",
    )
    trainer = pl.Trainer(
        accelerator="cuda",
        max_epochs=100,
        precision="16-mixed",
        callbacks=[progress_bar, checkpoint_callback],
    )

    # Train the model
    trainer.fit(lightning_model, train_loader,val_loader)

    best_ckpt = checkpoint_callback.best_model_path
    print(f"Best Lightning checkpoint: {best_ckpt}")

    lightning_model = Classifier.load_from_checkpoint(
        best_ckpt,
        model=model,  # supply same nn.Sequential structure
        optimizer_class=torch.optim.AdamW,
        optimizer_params={"lr": 1e-3}
    )

    # Test the model
    #trainer.test(lightning_model, test_loader)
    # Save model
    torch.save(model.state_dict(), model_path)
    print(f"Model saved to {model_path}")

In [ ]:
model.cuda().eval()

In [ ]:
from model.pointnet_plus import SAModule
def set_deterministic_fps(model, random_start=False):
    for module in model.modules():
        if isinstance(module, SAModule):
            module.random_start = random_start
            print(f"Set random_start={random_start} for {module.__class__.__name__}")

set_deterministic_fps(model)

In [ ]:
with torch.no_grad():
    model.eval()
    test_acc = 0
    for data, target in val_loader:
        data, target = data.cuda(), target.cuda()
        output = model(data)
        test_acc += output.argmax(dim=-1).eq(target).sum().item()
    test_acc /= len(dataset_val)
    print(f'Mean accuracy on the val set: {test_acc}.')

In [ ]:
with torch.no_grad():
    model.eval()
    test_acc = 0
    for data, target in test_loader:
        data, target = data.cuda(), target.cuda()
        output = model(data)
        test_acc += output.argmax(dim=-1).eq(target).sum().item()
    test_acc /= len(dataset_test)
    print(f'Mean accuracy on the transformed test set: {test_acc}.')

In [ ]:
modelpath = os.path.join(os.getcwd(), "model", f"{dataset}_metric2.pth")
import pytorch_lightning as pl
import os

model_path = modelpath

model_not_normalized2 = PointNetPlus()
model2 = torch.nn.Sequential(BatchNormalizeScale(), model_not_normalized2)
from dataset.geometric_wrapper import TensorGeometricModelWrapper
model2 = TensorGeometricModelWrapper(model2)

if os.path.exists(model_path) and True:
    print(f"Loading model from {model_path}")
    model2.load_state_dict(torch.load(model_path))
else:
    print(f"Training model and will save to {model_path}")

    # LightningModule wrapper
    lightning_model = Classifier(
        model2,
        optimizer_class=torch.optim.AdamW,
        optimizer_params={"lr": 1e-3}
    )

    # Callbacks
    progress_bar     = MyProgressBar()
    checkpoint_callback = ModelCheckpoint(
        monitor='val_acc',
        mode='max',
        save_top_k=1,
        verbose=True,
        dirpath=os.path.join(os.getcwd(), "model"),
        filename=f"{dataset}_val_acc_best"
    )

    # Trainer
    trainer = pl.Trainer(
        accelerator="cuda",
        max_epochs=200,
        precision="16-mixed",
        callbacks=[progress_bar, checkpoint_callback],
    )

    # Train
    trainer.fit(lightning_model, train_loader_transformed, val_loader_transformed)

    # 1) Reload LightningModule from best ckpt
    best_ckpt = checkpoint_callback.best_model_path
    print(f"Best Lightning checkpoint: {best_ckpt}")

    lightning_model = Classifier.load_from_checkpoint(
        best_ckpt,
        model=model2,  # supply same nn.Sequential structure
        optimizer_class=torch.optim.AdamW,
        optimizer_params={"lr": 1e-3}
    )

    # 2) Extract & save raw nn.Sequential weights
    model2.load_state_dict(lightning_model.model.state_dict())
    torch.save(model2.state_dict(), model_path)
    print(f"Best model saved to {model_path}")

set_deterministic_fps(model2, random_start=False)
with torch.no_grad():
    model2.eval().cuda()
    test_acc = 0
    for data, target in test_loader:
        data, target = data.cuda(), target.cuda()
        output = model2(data)
        test_acc += output.argmax(dim=-1).eq(target).sum().item()
    test_acc /= len(dataset_test)
    print(f'Mean accuracy on the transformed test set: {test_acc}.')


model2 = None
torch.cuda.empty_cache()

In [ ]:
list(model.named_modules())

In [ ]:
from confidence.utils import ModelInputWrapper, ModelOutputWrapper

#use model wrapper to extract the embeddings
dual_ouput_model = ModelInputWrapper(model,'model.1.mlp.5',flatten=True).eval()

In [ ]:
filter_correct = True  # Set to False to include all samples

embeddings = []
images = []
classes = []
logits = []

set_deterministic_fps(model, random_start=False)


for data, target in train_loader:
    data_cuda = data.cuda()
    emb_batch, logit_batch = dual_ouput_model(data_cuda)
    pred_batch = logit_batch.argmax(dim=-1).cpu().numpy()
    target_np = target.numpy()

    if filter_correct:
        mask = pred_batch == target_np
        emb_np = emb_batch.detach().cpu().numpy()[mask]
        logit_np = logit_batch.detach().cpu().numpy()[mask]
        img_np = data.numpy()[mask]
        class_np = target_np[mask]
    else:
        emb_np = emb_batch.detach().cpu().numpy()
        logit_np = logit_batch.detach().cpu().numpy()
        img_np = data.numpy()
        class_np = target_np

    embeddings.append(emb_np)
    logits.append(logit_np)
    images.append(img_np)
    classes.append(class_np)

set_deterministic_fps(model, random_start=True)

for i in range(5):
    for data, target in train_loader:
        data_cuda = data.cuda()
        emb_batch, logit_batch = dual_ouput_model(data_cuda)
        pred_batch = logit_batch.argmax(dim=-1).cpu().numpy()
        target_np = target.numpy()

        if filter_correct:
            mask = pred_batch == target_np
            emb_np = emb_batch.detach().cpu().numpy()[mask]
            logit_np = logit_batch.detach().cpu().numpy()[mask]
            img_np = data.numpy()[mask]
            class_np = target_np[mask]
        else:
            emb_np = emb_batch.detach().cpu().numpy()
            logit_np = logit_batch.detach().cpu().numpy()
            img_np = data.numpy()
            class_np = target_np

        embeddings.append(emb_np)
        logits.append(logit_np)
        images.append(img_np)
        classes.append(class_np)


set_deterministic_fps(model, random_start=False)






embeddings = np.vstack(embeddings)
logits     = np.vstack(logits)
images     = np.vstack(images)
classes    = np.hstack(classes)

# randomly sample 5000 entries
if embeddings.shape[0] > 100000:
    indices = np.random.choice(embeddings.shape[0], 100000, replace=False)
    embeddings_sampled = embeddings[indices]
    logits_sampled     = logits[indices]
    images_sampled     = images[indices]
    classes_sampled    = classes[indices]
else:
    embeddings_sampled = embeddings
    logits_sampled     = logits
    images_sampled     = images
    classes_sampled    = classes

In [ ]:
set_deterministic_fps(model, random_start=True)


In [ ]:
model.cuda().eval()

In [ ]:
import search.shgo
import importlib

importlib.reload(search.shgo)
di = search.shgo.SHGO(selection_method="topk", initial_samples=600, local_runs=2, local_max_steps=5)
di_no_grad = search.shgo.SHGO(selection_method="topk", local_max_steps=0, initial_samples=600)

In [ ]:
test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=2, shuffle=True, num_workers=4,persistent_workers=True)



In [ ]:
torch.cuda.empty_cache()

In [ ]:
rs = search.shgo.SHGO(selection_method="topk", initial_samples=240, local_runs=2, local_max_steps=0,
                      accept_only_if_improved=True, sigma=0)
from utils.eval.ood_performance import evaluate_confidence_and_search

from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence, PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import PerClassKNNConfidence

nn_pytorch = PerClassKNNConfidence(metric="cosine",k=2)
nn_pytorch.fit(torch.tensor(embeddings_sampled).cpu(), y=torch.tensor(classes_sampled).cpu())
nn_pytorch.cuda()

conf_split = PredictedSplitConfidence(nn_pytorch, EnergyConfidence(), mult=False, b=0.0000)
conf_mod_nn_pytorch = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_nn_pytorch = TransformationProblem(conf_mod_nn_pytorch, transform_seq, consolidate_method="consolidate_simple")

evaluate_confidence_and_search(model, rs, problem_nn_pytorch, test_loader, max_batch_override=32)

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
def evaluate_conf(model,di, problem, test_loader,max_batch_override=32,max_samples=100):
    if max_batch_override is not None:
        problem.max_batch_size = max_batch_override
    with torch.no_grad():
        test_acc_sim = 0
        counter = 0
        for data, target in tqdm.tqdm(test_loader):
            data, target = data.cuda(), target.cuda()
            with torch.enable_grad():
                res = di.optimize(problem, data.cuda(),y=target.cuda())
                res = list(res)
            torch.cuda.empty_cache()
            res[0] = res[0].detach()  # Detach the result to avoid gradients
            res[1] = res[1].detach()  # Detach the result to avoid gradients
            # Apply the transformation and get predictions
            x_transformed2 = problem.transform(data.cuda(), res[0])
            logits = model(x_transformed2)
            output = logits.argmax(dim=-1)
            test_acc_sim += output.eq(target).sum().cpu().detach().item()
            counter += output.shape[0]
            if max_samples is not None and counter >= max_samples:
                break

    test_acc_sim /= counter
    return test_acc_sim

In [ ]:
import tqdm

from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence

conf_mod_nn_pytorch = SinglePassConfidence(model,EnergyConfidence())
problem_nn_pytorch = TransformationProblem(conf_mod_nn_pytorch,transform_seq,consolidate_method="consolidate_simple")

evaluate_confidence_and_search(model,rs,problem_nn_pytorch,val_loader_transformed,max_batch_override=32)


In [ ]:
#draw here
problem = problem_nn_pytorch
with torch.no_grad():
    data,target = next(iter(test_loader))
    data, target = data.cuda(), target.cuda()
    #plot data zero
    index =1
    plot_interactive_3d(data[index].cpu().numpy())

    with torch.enable_grad():
        res = di.optimize(problem_nn_pytorch, data.cuda(),y=target.cuda())
        res = list(res)

    torch.cuda.empty_cache()
    res[0] = res[0].detach()  # Detach the result to avoid gradients
    res[1] = res[1].detach()  # Detach the result to avoid gradients

    # Apply the transformation and get
    x_transformed2 = problem.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    plot_interactive_3d(x_transformed2[index].cpu().numpy())
    print(f"Object 1 class: {target[index].item()}, predicted class: {output[index].item()}")



In [ ]:
import tqdm

from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence
nn_pytorch = KNNConfidence(metric="cosine",k=3)
nn_pytorch.fit(torch.tensor(embeddings_sampled).cpu())
nn_pytorch.cuda()


conf_split = SplitConfidence(nn_pytorch,EnergyConfidence(), mult=True,b=0.0,scale_final="softplus")
conf_mod_nn_pytorch = SinglePassConfidence(dual_ouput_model,conf_split,index=1)
problem_nn_pytorch = TransformationProblem(conf_mod_nn_pytorch,transform_seq,consolidate_method="consolidate_simple")
evaluate_confidence_and_search(model,di,problem_nn_pytorch,val_loader_transformed,max_batch_override=32)


In [ ]:
import tqdm

from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence
nn_pytorch = KNNConfidence(metric="cosine",k=3)
nn_pytorch.fit(torch.tensor(embeddings_sampled).cpu())
nn_pytorch.cuda()


conf_split = SplitConfidence(nn_pytorch,EnergyConfidence(), mult=False,b=0.0001)
conf_mod_nn_pytorch = SinglePassConfidence(dual_ouput_model,conf_split,index=1)
problem_nn_pytorch = TransformationProblem(conf_mod_nn_pytorch,transform_seq,consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_nn_pytorch,test_loader)


In [ ]:
import tqdm

from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence
nn_pytorch = KNNConfidence(metric="cosine",k=3)
nn_pytorch.fit(torch.tensor(embeddings_sampled).cpu())
nn_pytorch.cuda()


conf_split = SplitConfidence(nn_pytorch,EnergyConfidence(), mult=False,b=0.1)
conf_mod_nn_pytorch = SinglePassConfidence(dual_ouput_model,conf_split,index=1)
problem_nn_pytorch = TransformationProblem(conf_mod_nn_pytorch,transform_seq,consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_nn_pytorch,test_loader)


In [ ]:
import tqdm

from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence
nn_pytorch = KNNConfidence(metric="cosine",k=3)
nn_pytorch.fit(torch.tensor(embeddings_sampled).cpu())
nn_pytorch.cuda()


conf_split = SplitConfidence(nn_pytorch,EnergyConfidence(), mult=False,b=1)
conf_mod_nn_pytorch = SinglePassConfidence(dual_ouput_model,conf_split,index=1)
problem_nn_pytorch = TransformationProblem(conf_mod_nn_pytorch,transform_seq,consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_nn_pytorch,test_loader)


In [ ]:
import tqdm

from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence
nn_pytorch = KNNConfidence(metric="cosine",k=3)
nn_pytorch.fit(torch.tensor(embeddings_sampled).cpu())
nn_pytorch.cuda()


conf_split = SplitConfidence(nn_pytorch,EnergyConfidence(), mult=False,b=10)
conf_mod_nn_pytorch = SinglePassConfidence(dual_ouput_model,conf_split,index=1)
problem_nn_pytorch = TransformationProblem(conf_mod_nn_pytorch,transform_seq,consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_nn_pytorch,test_loader)


In [ ]:
import tqdm

from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence
nn_pytorch = KNNConfidence(metric="cosine",k=3)
nn_pytorch.fit(torch.tensor(embeddings_sampled).cpu())
nn_pytorch.cuda()


conf_split = SplitConfidence(nn_pytorch,EnergyConfidence(), mult=False,b=0.0)
conf_mod_nn_pytorch = SinglePassConfidence(dual_ouput_model,conf_split,index=1)
problem_nn_pytorch = TransformationProblem(conf_mod_nn_pytorch,transform_seq,consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_nn_pytorch,test_loader)


In [ ]:
import tqdm

from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence
nn_pytorch = KNNConfidence(metric="cosine",k=3)
nn_pytorch.fit(torch.tensor(embeddings).cpu())
nn_pytorch.cuda()


conf_split = SplitConfidence(nn_pytorch,EnergyConfidence(), mult=False,b=0.0000)
conf_mod_nn_pytorch = SinglePassConfidence(dual_ouput_model,conf_split,index=1)
problem_nn_pytorch = TransformationProblem(conf_mod_nn_pytorch,transform_seq,consolidate_method="consolidate_simple")

evaluate_conf(model,di,problem_nn_pytorch,test_loader)

In [ ]:


from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence,PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import PerClassKNNConfidence
nn_pytorch = PerClassKNNConfidence(metric="cosine")
nn_pytorch.fit(torch.tensor(embeddings_sampled).cpu(), y=torch.tensor(classes_sampled).cpu())
nn_pytorch.cuda()


conf_split = PredictedSplitConfidence(nn_pytorch,EnergyConfidence(), mult=False,b=0.0)
conf_mod_nn_pytorch = SinglePassConfidence(dual_ouput_model,conf_split,index=1)
problem_nn_pytorch = TransformationProblem(conf_mod_nn_pytorch,transform_seq,consolidate_method="consolidate_simple")

evaluate_conf(model,di,problem_nn_pytorch,test_loader)

In [ ]:
import tqdm

from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence
nn_pytorch = PerClassKNNConfidence(metric="cosine")
nn_pytorch.fit(torch.tensor(embeddings_sampled).cpu(), y=torch.tensor(classes_sampled).cpu())
nn_pytorch.cuda()



conf_split = PredictedSplitConfidence(nn_pytorch,EnergyConfidence(), mult=False,b=1)
conf_mod_nn_pytorch = SinglePassConfidence(dual_ouput_model,conf_split,index=1)
problem_nn_pytorch = TransformationProblem(conf_mod_nn_pytorch,transform_seq,consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_nn_pytorch,test_loader)


In [ ]:


from confidence.direct.prob_based import EntropyConfidence
from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence,NNGuideSplitConfidence


class one_confidence(nn.Module):
    def __init__(self):
        super().__init__()
        self.EnergyConfidence = EnergyConfidence()

    def forward(self, x,y=None):
        return torch.ones_like(x)[:,0]

class one_confidence2(nn.Module):
    def __init__(self):
        super().__init__()
        self.EnergyConfidence = EnergyConfidence()

    def forward(self, x,y=None):
        return self.EnergyConfidence(x,y)

conf_split = NNGuideSplitConfidence(one_confidence2(), k=3)
conf_split.fit((torch.tensor(embeddings_sampled).cpu(), torch.tensor(logits_sampled).cpu()))
conf_split.cuda()
conf_mod_nn_pytorch = SinglePassConfidence(dual_ouput_model,conf_split,index=1)
problem_nn_pytorch = TransformationProblem(conf_mod_nn_pytorch,transform_seq,consolidate_method="consolidate_simple")

evaluate_conf(model,di,problem_nn_pytorch,test_loader)

In [ ]:
nn_pytorch.save("test.pth")
nn_pytorch.load("test.pth")

In [ ]:
evaluate_conf(model,di,problem_nn_pytorch,test_loader)


In [ ]:
from confidence.unsupervised.classic.lle import LLEResidualConfidence

lle_conf = LLEResidualConfidence()
lle_conf.fit(torch.tensor(embeddings_sampled).cpu())
lle_conf.cuda()
conf_split = SplitConfidence(lle_conf, EnergyConfidence(), mult=False, b=0.0)
conf_mod_lle = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
transformation_problem = TransformationProblem(conf_mod_lle, transform_seq, consolidate_method="consolidate_simple")
evaluate_conf(model,di,transformation_problem,test_loader)


In [ ]:
from confidence.unsupervised.classic.openmax import OpenMaxConfidence
from confidence.control.split import PredictedSplitConfidence

lle_conf = OpenMaxConfidence()
lle_conf.fit(torch.tensor(embeddings_sampled).cuda(), y=torch.tensor(classes_sampled).cuda())
lle_conf.cuda()
conf_split = PredictedSplitConfidence(lle_conf, EnergyConfidence(), mult=False, b=0.0)
conf_mod_lle = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
transformation_problem = TransformationProblem(conf_mod_lle, transform_seq, consolidate_method="consolidate_simple")

evaluate_conf(model,di,transformation_problem,test_loader)


In [ ]:
#extract logits from train
logits2 = []
classes2 = []
for batch in train_loader:
    logits_batch = model(batch[0].cuda())
    logits2.append(logits_batch.detach().cpu().numpy())
    classes2.append(batch[1].detach().cpu().numpy())
logits2 = np.vstack(logits2)
classes2 = np.hstack(classes2)

In [ ]:
from confidence.unsupervised.classic.openmax import OpenMaxConfidence
from confidence.control.split import PredictedSplitConfidence

lle_conf2 = OpenMaxConfidence()
lle_conf2.fit(torch.tensor(logits2).cuda(), y=torch.tensor(classes2).cuda())
lle_conf2.cuda()
conf_split = PredictedSplitConfidence(lle_conf,lle_conf2, mult=False, a=0.0,b=1.0)
conf_mod_lle = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
transformation_problem = TransformationProblem(conf_mod_lle, transform_seq, consolidate_method="consolidate_simple")

evaluate_conf(model,di,transformation_problem,test_loader)

In [ ]:
from confidence.unsupervised.classic.openmax3 import OpenMaxConfidence as OpenMaxConfidence3
from confidence.control.split import PredictedSplitConfidence
import tqdm
lle_conf3 = OpenMaxConfidence3()
lle_conf3.fit(torch.tensor(logits2).cuda(), y=torch.tensor(classes2).cuda())
lle_conf3.cuda()

conf_mod_lle = SinglePassConfidence(model, lle_conf3)
transformation_problem = TransformationProblem(conf_mod_lle, transform_seq, consolidate_method="consolidate_simple")
evaluate_conf(model,di,transformation_problem,test_loader)

In [ ]:
from confidence.unsupervised.classic.nn import NNDistanceConfidence

nearest_neighbor_confidence = NNDistanceConfidence(index_type="ivfpq", number_of_neighbors=3)
nearest_neighbor_confidence.fit(torch.tensor(embeddings_sampled).cpu(), y=torch.tensor(classes_sampled).cpu())
nearest_neighbor_confidence.cuda()
conf_split = SplitConfidence(nearest_neighbor_confidence, EnergyConfidence(), mult=False, b=0.0)
conf_mod = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
transformation_problem = TransformationProblem(conf_mod, transform_seq, consolidate_method="consolidate_simple")

evaluate_conf(model,di,transformation_problem,test_loader)

In [ ]:

from confidence.unsupervised.classic.gmm import GaussianMixtureConfidence

gmm_conf = GaussianMixtureConfidence(n_components=10, covariance_type="full")
gmm_conf.fit(embeddings_sampled)
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")

evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:

from confidence.unsupervised.classic.isolation import HardIsolationForestConfidence

gmm_conf = HardIsolationForestConfidence(contamination=0.01,max_features=10)
gmm_conf.fit(embeddings)
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")

evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:


from confidence.unsupervised.classic.trust_score import TrustScoreTorchConfidence

gmm_conf = TrustScoreTorchConfidence()
gmm_conf.device = "cuda"
gmm_conf.fit(torch.tensor(embeddings_sampled).cuda(),torch.tensor(classes_sampled).cuda())
gmm_conf.cuda()
conf_split = PredictedSplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")

evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:


from confidence.unsupervised.classic.kde import KDEConfidence

gmm_conf = KDEConfidence(bandwidth=0.2, kernel="gaussian")
gmm_conf.fit(embeddings_sampled)
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")

evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:
import confidence.unsupervised.classic.pca
import importlib

importlib.reload(confidence.unsupervised.classic.pca)
from confidence.unsupervised.classic.pca import PCATorchConfidence

gmm_conf = PCATorchConfidence(n_components=32)
gmm_conf.fit(torch.tensor(embeddings).cuda())
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")

evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:




from confidence.unsupervised.classic.gaussian import ImageGaussianConfidence

gmm_conf = ImageGaussianConfidence()
gmm_conf.fit(torch.tensor(embeddings).cuda())
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")

evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:


#from confidence.unsupervised.classic.dice import DICEConfidence


#gmm_conf = DICEConfidence(model,percentile=0.95)
#gmm_conf.fit(torch.tensor(embeddings).cuda(),y=torch.tensor(classes).cuda())
#gmm_conf.cuda()
#conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
#gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
#problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")

#evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:


#from confidence.unsupervised.classic.dice import DICEConfidence


#gmm_conf = DICEConfidence(model,percentile=0.01)
#gmm_conf.fit(torch.tensor(embeddings).cuda(),y=torch.tensor(classes).cuda())
#gmm_conf.cuda()
#conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
#gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
#problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")

#evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:






from confidence.unsupervised.classic.kl_matching import KLMatchingConfidence

gmm_conf = KLMatchingConfidence()
gmm_conf.fit(torch.tensor(logits2).cuda(),y=torch.tensor(classes2).cuda())
gmm_conf.cuda()
gmm_conf = SinglePassConfidence(model, gmm_conf)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
print("fitted")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:
gmm_conf = LOFTorchConfidence()
gmm_conf.fit(torch.tensor(embeddings).cuda(),y=torch.tensor(classes).cuda())
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
print("fitted")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:
from confidence.unsupervised.classic.mahalanobis_relative import RelativeMahalanobisConfidence

gmm_conf = RelativeMahalanobisConfidence()
gmm_conf.fit(torch.tensor(embeddings).cuda(),y=torch.tensor(classes).cuda())
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
print("fitted")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:
torch.cuda.empty_cache()

In [ ]:
from confidence.unsupervised.classic.pyood_wrapper import MahalanobisConfidence

gmm_conf = MahalanobisConfidence()
gmm_conf.fit(torch.tensor(embeddings).cuda(),y=torch.tensor(classes).cuda())
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
print("fitted")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:

from confidence.unsupervised.classic.mtc import ManifoldTangentConfidence

gmm_conf = ManifoldTangentConfidence(n_components=128,n_clusters=4)
gmm_conf.fit(torch.tensor(embeddings).cuda())
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
print("fitted")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:

from confidence.control.classify import ClassifyingConfidence
from confidence.unsupervised.classic.VIM import ViMTorchConfidence
gmm_conf = SHETorchConfidence()
gmm_conf.fit(torch.tensor(embeddings).cuda(),y=torch.tensor(classes).cuda())
gmm_conf.cuda()
conf_split = ClassifyingConfidence(gmm_conf,index=1,index_confidence=0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
print("fitted")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:

from confidence.control.classify import ClassifyingConfidence
from confidence.unsupervised.classic.VIM import ViMTorchConfidence
import importlib
importlib.reload(confidence.unsupervised.classic.VIM)
gmm_conf = ViMTorchConfidence(model.eval(),n_dim=32,use_energy=True)
gmm_conf.fit(torch.tensor(embeddings).cuda(),y=torch.tensor(classes).cuda())
gmm_conf.cuda()
conf_split = ClassifyingConfidence(gmm_conf,index=1,index_confidence=0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
print("fitted")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:

from confidence.control.classify import ClassifyingConfidence
from confidence.unsupervised.classic.VIM import ViMTorchConfidence
import importlib
importlib.reload(confidence.unsupervised.classic.VIM)
gmm_conf = ViMTorchConfidence(model.eval(),n_dim=128,use_energy=True)
gmm_conf.fit(torch.tensor(embeddings).cuda(),y=torch.tensor(classes).cuda())
gmm_conf.cuda()
conf_split = ClassifyingConfidence(gmm_conf,index=1,index_confidence=0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
print("fitted")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:
#reload tqdm
import tqdm
import importlib
importlib.reload(tqdm)

In [ ]:
import confidence.unsupervised.ml.autoencoder
import importlib

importlib.reload(confidence.unsupervised.ml.autoencoder)
from confidence.unsupervised.ml.autoencoder import BasicAutoencoderConfidence

encoder_lin = torch.nn.Sequential(
    nn.Linear(embeddings.shape[1], 512),
    nn.GELU(),
    nn.Linear(512, 512),
    nn.GELU(),
    nn.Linear(512, 128)
).cuda()
decoder_lin = torch.nn.Sequential(
    nn.Linear(128, 512),
    nn.GELU(),
    nn.Linear(512, 512),
    nn.GELU(),
    nn.Linear(512, embeddings.shape[1])
).cuda()
gmm_conf = BasicAutoencoderConfidence(encoder_lin, decoder_lin,trainer_kwargs={"max_epochs": 30},dataloader_kwargs={"batch_size": 128})
gmm_conf.fit(torch.tensor(embeddings).cuda())
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:
import confidence.unsupervised.ml.autoencoder
import importlib

importlib.reload(confidence.unsupervised.ml.autoencoder)
from confidence.unsupervised.ml.autoencoder import ConditionalAutoencoderConfidence

encoder_lin = torch.nn.Sequential(
    nn.Linear(embeddings.shape[1], 512),
    nn.GELU(),
    nn.Linear(512, 512),
    nn.GELU(),
    nn.Linear(512, 128)
).cuda()
decoder_lin = torch.nn.Sequential(
    nn.Linear(138, 512),
    nn.GELU(),
    nn.Linear(512, 512),
    nn.GELU(),
    nn.Linear(512, embeddings.shape[1])
).cuda()
gmm_conf = ConditionalAutoencoderConfidence(encoder_lin, decoder_lin,num_classes=10,trainer_kwargs={"max_epochs": 5},dataloader_kwargs={"batch_size": 128})
gmm_conf.fit(torch.tensor(embeddings).cuda(),torch.tensor(classes).cuda())
gmm_conf.cuda()
conf_split = PredictedSplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:


from confidence.unsupervised.ml.deepsvd import DeepSVDDConfidence

encoder_lin_deep = torch.nn.Sequential(
    nn.Linear(embeddings.shape[1], 128,bias=False),
    nn.LeakyReLU(),
    nn.Linear(128, 128,bias=False),
    nn.LeakyReLU(),
    nn.Linear(128, 16,bias=False)
).cuda()
gmm_conf = DeepSVDDConfidence(encoder_lin_deep,objective="one-class",trainer_kwargs={"max_epochs": 20},dataloader_kwargs={"batch_size": 128},weight_decay=0.001)
gmm_conf.cuda()
gmm_conf.fit(torch.tensor(embeddings).cuda()).cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:
from confidence.unsupervised.ml.vae import VAEConfidence
import confidence.unsupervised.ml.autoencoder
import importlib

importlib.reload(confidence.unsupervised.ml.autoencoder)


encoder_lin2 = torch.nn.Sequential(
    nn.Linear(embeddings.shape[1], 512),
    nn.GELU(),
    nn.Linear(512, 128),
    nn.GELU(),
).cuda()

class VAEEncoder(torch.nn.Module):
    def __init__(self,model,indim=128,outdim=16):
        super(VAEEncoder, self).__init__()
        self.model = model
        self.lin1= nn.Linear(indim, outdim)
        self.lin2 = nn.Linear(indim, outdim)
    def forward(self, x):
        x = self.model(x)
        return self.lin1(x), self.lin2(x)

encoder_lin2 = VAEEncoder(encoder_lin2).cuda()




decoder_lin2 = torch.nn.Sequential(
    nn.Linear(16, 128),
    nn.GELU(),
    nn.Linear(128, 512),
    nn.GELU(),
    nn.Linear(512, embeddings.shape[1])
).cuda()
gmm_conf = VAEConfidence(encoder_lin2, decoder_lin2,latent_dim=16,mode="elbo",beta=0.1,
                         trainer_kwargs={"max_epochs": 15},
                         dataloader_kwargs={"batch_size": 128})
gmm_conf.fit(torch.tensor(embeddings).cuda())
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:
from confidence.unsupervised.ml.flow import FlowConfidence
import normflows as nf
# Define 2D Gaussian base distribution
base = nf.distributions.base.DiagGaussian(embeddings.shape[1])

# Define list of flows
num_layers = 2
flows = []
for i in range(num_layers):
    # Neural network with two hidden layers having 64 units each
    # Last layer is initialized by zeros making training more stable
    param_map = nf.nets.MLP([embeddings.shape[1]//2, embeddings.shape[1], embeddings.shape[1], embeddings.shape[1]], init_zeros=True)
    # Add flow layer
    flows.append(nf.flows.AffineCouplingBlock(param_map))
    # Swap dimensions
    flows.append(nf.flows.Permute(2, mode='swap'))

flow = nf.NormalizingFlow(base, flows)

gmm_conf = FlowConfidence(flow, None,
                         trainer_kwargs={"max_epochs": 20},
                         dataloader_kwargs={"batch_size": 128})
gmm_conf.fit(torch.tensor(embeddings).cuda())
gmm_conf.cuda()


In [ ]:
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:
class PointNetPlusWrapper1(torch.nn.Module):
    def __init__(self,pointnet_cls):
        super().__init__()
        self.pointnet_cls = pointnet_cls

    def forward(self, data):
        sa0_out = (data.x, data.pos, data.batch)
        sa1_out = self.pointnet_cls.sa1_module(*sa0_out)
        sa2_out = self.pointnet_cls.sa2_module(*sa1_out)
        sa3_out = self.pointnet_cls.sa3_module(*sa2_out)
        x, pos, batch = sa3_out

        return x

In [ ]:
class PointNetPlusWrapper2(torch.nn.Module):
    def __init__(self,pointnet_cls):
        super().__init__()
        self.pointnet_cls = pointnet_cls

    def forward(self, x):
        return self.pointnet_cls.mlp(x)

In [ ]:
pt = model.model[1]

In [ ]:
backbone = PointNetPlusWrapper1(pt).cuda()
backbone = torch.nn.Sequential(
    model.model[0],
    backbone,
).cuda()
backbone = TensorGeometricModelWrapper(backbone).cuda()
head = PointNetPlusWrapper2(pt).cuda()

In [ ]:
import confidence.model.ash
import importlib
importlib.reload(confidence.model.ash)
from confidence.model.ash import ReActConfidence
#TODO apply this to the last layer only.Need to retrain model as mlp in torchgemetrics is difficult to split into parts as it is not a sequential model.
gmm_conf = ReActConfidence(backbone,head,threshold=3.0)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")

evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:
list(model.named_modules())

In [ ]:
dual_output_model3 = ModelInputWrapper(model,'model.1.mlp',flatten=True).eval()

In [ ]:
filter_correct = True  # Set to False to include all samples

embeddings2 = []
classes2 = []

for data, target in train_loader:
    data_cuda = data.cuda()
    emb_batch, logit_batch = dual_output_model3(data_cuda)
    pred_batch = logit_batch.argmax(dim=-1).cpu().numpy()
    target_np = target.numpy()

    if filter_correct:
        mask = pred_batch == target_np
        emb_np = emb_batch.detach().cpu().numpy()[mask]
        logit_np = logit_batch.detach().cpu().numpy()[mask]
        img_np = data.numpy()[mask]
        class_np = target_np[mask]
    else:
        emb_np = emb_batch.detach().cpu().numpy()
        logit_np = logit_batch.detach().cpu().numpy()
        img_np = data.numpy()
        class_np = target_np

    embeddings2.append(emb_np)
    classes2.append(class_np)

embeddings2 = np.vstack(embeddings2)
classes2    = np.hstack(classes2)


In [ ]:
embeddings2.shape

In [ ]:
list(backbone.named_modules())

In [ ]:
import confidence.model.ash
import importlib
importlib.reload(confidence.model.ash)
from confidence.model.ash import ReActConfidence
knn_conf = PerClassKNNConfidence(k=3,metric="cosine")
knn_conf.fit(torch.tensor(embeddings2).cuda(), y=torch.tensor(classes2).cuda())
conf = SplitConfidence(knn_conf, EnergyConfidence(), mult=True, b=0.0)
gmm_conf = SinglePassConfidence(dual_output_model3,index=1,confidence=conf)




problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:
import confidence.model.ash
import importlib
importlib.reload(confidence.model.ash)
from confidence.model.ash import ReActConfidence
knn_conf = PerClassKNNConfidence(k=3,metric="euclidean")
knn_conf.fit(torch.tensor(embeddings2).cuda(), y=torch.tensor(classes2).cuda())
conf = PredictedSplitConfidence(knn_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_output_model3,index=1,confidence=conf)




problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:
import confidence.model.ash
import importlib
importlib.reload(confidence.model.ash)
from confidence.model.ash import ReActConfidence
knn_conf = PerClassKNNConfidence(k=3,metric="euclidean")
knn_conf.fit(torch.tensor(embeddings).cuda(), y=torch.tensor(classes).cuda())
conf = PredictedSplitConfidence(knn_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model,index=1,confidence=conf)




problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:
import confidence.model.ash
import importlib
importlib.reload(confidence.model.ash)
from confidence.model.ash import ReActConfidence
knn_conf = PerClassKNNConfidence(k=3,metric="cosine")
knn_conf.fit(torch.tensor(embeddings).cuda(), y=torch.tensor(classes).cuda())
conf = PredictedSplitConfidence(knn_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model,index=1,confidence=conf)




problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:
from confidence.input_transform import InputTransform

input_transform = InputTransform(standardize=True)

In [ ]:
import confidence.model.ash
import importlib
importlib.reload(confidence.model.ash)
from confidence.model.ash import ReActConfidence
knn_conf = PerClassKNNConfidence(k=3,metric="euclidean",input_transform=input_transform)
knn_conf.fit(torch.tensor(embeddings).cuda(), y=torch.tensor(classes).cuda())
conf = PredictedSplitConfidence(knn_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model,index=1,confidence=conf)




problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:
import confidence.model.ash
import importlib
importlib.reload(confidence.model.ash)
from confidence.model.ash import ReActConfidence
knn_conf = PerClassKNNConfidence(k=3,metric="cosine",input_transform=input_transform)
knn_conf.fit(torch.tensor(embeddings).cuda(), y=torch.tensor(classes).cuda())
conf = PredictedSplitConfidence(knn_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model,index=1,confidence=conf)




problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:
import confidence.model.ash
import importlib
importlib.reload(confidence.model.ash)
from confidence.model.ash import ReActConfidence
knn_conf = PerClassKNNConfidence(k=3,metric="cosine",input_transform=input_transform)
knn_conf.fit(torch.tensor(embeddings).cuda(), y=torch.tensor(classes).cuda())
conf = PredictedSplitConfidence(knn_conf, EnergyConfidence(), mult=False, b=0.001)
gmm_conf = SinglePassConfidence(dual_ouput_model,index=1,confidence=conf)




problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:
import confidence.model.ash
import importlib
importlib.reload(confidence.model.ash)
from confidence.model.ash import ReActConfidence
knn_conf = PerClassKNNConfidence(k=3,metric="cosine",input_transform=input_transform)
knn_conf.fit(torch.tensor(embeddings).cuda(), y=torch.tensor(classes).cuda())
conf = PredictedSplitConfidence(knn_conf, EnergyConfidence(), mult=False, b=0.0001)
gmm_conf = SinglePassConfidence(dual_ouput_model,index=1,confidence=conf)




problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_gmm_conf,test_loader)

In [ ]:
import confidence.model.ash
import importlib
importlib.reload(confidence.model.ash)
from confidence.model.ash import ReActConfidence
knn_conf = PerClassKNNConfidence(k=3,metric="cosine",input_transform=input_transform)
knn_conf.fit(torch.tensor(embeddings).cuda(), y=torch.tensor(classes).cuda())
conf = PredictedSplitConfidence(knn_conf, EnergyConfidence(), mult=False, b=0.00001)
gmm_conf = SinglePassConfidence(dual_ouput_model,index=1,confidence=conf)




problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
evaluate_conf(model,di,problem_gmm_conf,test_loader)